In [1]:
## Load libraries
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from keras.datasets import mnist
plt.style.use('dark_background')
%matplotlib inline

In [2]:
np.set_printoptions(precision=2)


In [3]:
import tensorflow as tf

In [4]:
## Load MNIST data
(X_train, _), (X_test, _) = mnist.load_data()
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1]*X_train.shape[2])
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1]*X_test.shape[2])

In [5]:
print(X_train.shape)
print(X_test.shape)

(60000, 784)
(10000, 784)


In [6]:
num_features = X_train.shape[1]
num_samples = X_train.shape[0]


# Normalize the samples (images)
xmax = np.amax(X_train)
xmin = np.amin(X_train)
X_train = (X_train - xmin) / (xmax - xmin) # all train features turn into a number between 0 and 1
X_test = (X_test - xmin)/(xmax - xmin)

print('MNIST set')
print('---------------------')
print('Number of training samples = %d', num_samples)
print('Number of features = %d'%(num_features))

MNIST set
---------------------
Number of training samples = %d 60000
Number of features = 784


In [7]:
#Parameters for the autoencoder
batch_size = 256
max_epochs = 50
learning_rate = 1e-3
latent_dim = 128
hidden_dim = 256
original_dim = X_train.shape[1]

In [8]:
type(X_train)

numpy.ndarray

In [9]:
#Covert numpy to tf.data.Dataset
training_dataset = tf.data.Dataset.from_tensor_slices(X_train).batch(batch_size)

In [10]:
#Encoder
class Encoder(tf.keras.layers.Layer):
    #Define input independent model information
    def __init__(self, hidden_dim, latent_dim):
        super(Encoder, self).__init__()
        self.encoder_layer1 = tf.keras.layers.Dense(units=hidden_dim, activation=tf.nn.relu) #hidden layer
        self.encoder_layer2 = tf.keras.layers.Dense(units=latent_dim, activation=tf.nn.relu) #latent layer

    #Method for forward propagation
    def call(self, input_features):
        a = self.encoder_layer1(input_features)
        a = self.encoder_layer2(a)
        return a

In [11]:
#Decoder
class Decoder(tf.keras.layers.Layer):
    def __init__(self, latent_dim, hidden_dim, original_dim):
        super(Decoder, self).__init__()
        self.decoder_layer1 = tf.keras.layers.Dense(units=hidden_dim, activation=tf.nn.relu)
        self.decoder_layer2 = tf.keras.layers.Dense(units=original_dim, activation=tf.nn.relu)

    def call(self, encoded_feature):
        a = self.decoder_layer1(encoded_feature)
        a = self.decoder_layer2(a)
        return a


In [12]:
#Autoencoder
class Autoencoder(tf.keras.Model):
    def __init__(self, latent_dim, hidden_dim, original_dim):
        super(Autoencoder, self).__init__()
        self.loss = []
        self.encoder = Encoder(hidden_dim = hidden_dim, latent_dim = latent_dim)
        self.decoder = Decoder(latent_dim = latent_dim, hidden_dim = hidden_dim, original_dim = original_dim)

    def call(self, input_features):
        encoded_features = self.encoder(input_features)
        reconstructed_features = self.decoder(encoded_features)
        return reconstructed_features

In [13]:
#Build the model
autoencoder = Autoencoder(latent_dim = latent_dim, hidden_dim = hidden_dim, original_dim = original_dim)    

In [14]:
opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)

In [15]:
#Custom training
def loss(true, pred):
    true = tf.cast(true, tf.float32)
    pred = tf.cast(pred, tf.float32)    
    return tf.reduce_mean(tf.square(tf.subtract(true, pred)))

In [16]:
#Custom training - compute gradient of loss and update weights
@tf.function
def train(loss, model, opt, original_features):
    with tf.GradientTape() as g:
        pred = model(original_features)
        reconstruction_error = loss(original_features, pred)
    gradients = g.gradient(reconstruction_error, model.trainable_variables)
    gradients_variables = zip(gradients, model.trainable_variables)
    opt.apply_gradients(gradients_variables)

    return reconstruction_error

In [17]:
#Train network by iterating epochs
# Varible to store training loss per epoch
loss_train_epoch = tf.keras.metrics.Mean()
for epoch in range(max_epochs):
    for step, train_batch_features in enumerate(training_dataset):
        loss_batch =  train(loss, autoencoder, opt, train_batch_features)

        # Append training loss
        loss_train_epoch(loss_batch)
    print('train loss = %f'%(loss_train_epoch.result()))



train loss = 0.026401
train loss = 0.018935
train loss = 0.015858
train loss = 0.014107
train loss = 0.012942
train loss = 0.012099
train loss = 0.011461
train loss = 0.010951
train loss = 0.010538
train loss = 0.010193
train loss = 0.009902
train loss = 0.009649
train loss = 0.009430
train loss = 0.009237
train loss = 0.009067
train loss = 0.008913
train loss = 0.008774
train loss = 0.008650
train loss = 0.008534
train loss = 0.008429
train loss = 0.008333
train loss = 0.008244
train loss = 0.008161
train loss = 0.008084
train loss = 0.008012
train loss = 0.007943
train loss = 0.007879
train loss = 0.007820
train loss = 0.007763
train loss = 0.007711
train loss = 0.007662
train loss = 0.007614
train loss = 0.007569
train loss = 0.007526
train loss = 0.007487
train loss = 0.007448
train loss = 0.007412


In [ ]:
# train_loop(autoencoder, opt, loss, training_dataset, epochs = max_epochs)

In [ ]:
# Plot the original and decoded images
original_images = X_test[:10]
decoded_images = autoencoder(original_images)

# Plot the original and decoded images
fig, axes = plt.subplots(nrows=2, ncols=10, figsize=(20, 4))
for i in range(10):
    axes[0, i].imshow(original_images[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(decoded_images[i].numpy().reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')

plt.suptitle('Original and Decoded Images')
plt.show()


In [ ]:
#Plot some input images and their reconstructed versions
nimages = 10
fig, ax = plt.subplots(figsize=(10, 4))
for ind in range(nimages):
    #original image
    ax = plt.subplot(2, nimages, ind + 1)
    plt.imshow(X_test[ind].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    #reconstructed image
    ax = plt.subplot(2, nimages, ind + 1 + nimages)
    plt.imshow(decoded_images[ind].numpy().reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
plt.show()  

In [ ]:
# Plot the original and decoded images
original_images = X_test[:10]
decoded_images = autoencoder(original_images)

# Plot the original and decoded images
fig, axes = plt.subplots(nrows=2, ncols=10, figsize=(20, 4))
for i in range(10):
    axes[0, i].imshow(original_images[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(decoded_images[i].numpy().reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')

plt.suptitle('Original and Decoded Images')
plt.show()
